# YouTube → Untertitel & Transkript

**Version 1.3** · Zwei Abrufwege für vorhandene Untertitel: zuerst direkt, dann mit yt-dlp. Fehler zeigen jetzt die genaue Ursache je Abrufweg. Bei veraltetem Live-Status **Stream_bereits_beendet** anhaken.

**Link einfügen. ▶ drücken. Dateien herunterladen.** Funktioniert im Browser auf Handy, Tablet und PC.

1. Im Feld **YouTube_Link** den Videolink einfügen.
2. Links neben der Zelle **▶** drücken. Beim ersten Start gegebenenfalls mit Google anmelden und „Trotzdem ausführen“ bestätigen.
3. Anschließend **ZIP, SRT, VTT oder TXT** herunterladen. Für das nächste Video Link austauschen und erneut ▶ drücken.

**Standard:** vorhandene Untertitel zuerst mit `youtube-transcript-api`, danach bei Bedarf mit yt-dlp abrufen. Die Originalsprache wird bevorzugt; falls sie nicht ermittelbar ist, wird das gekennzeichnet. Falls kein Untertitelabruf gelingt und die Tonspur zugänglich ist, Sprache mit Whisper erkennen. Kein API-Key. Für vorhandene Untertitel reicht die normale CPU-Laufzeit.

**Gerade beendeter Stream:** Der `/live/`-Link bleibt gültig. Wenn YouTube weiterhin „live“ meldet, **Stream_bereits_beendet** aktivieren. Das Tool versucht vorhandene Untertitel und eine fertige Audiodatei abzurufen. YouTube kann die Aufzeichnung und die neuen automatischen Untertitel noch verarbeiten; dann meldet das Tool das gesondert. Es startet keine dauerhafte Liveaufnahme und gibt keinen erkennbar unvollständigen Audioausschnitt als ganzes Video aus.

**Sehr lange Videos ohne Untertitel:** vor dem Start bei Bedarf **Laufzeit → Laufzeittyp ändern → T4 GPU** wählen; eine A100 ist nicht erforderlich. Für dauerhafte Zwischenstände **In_Google_Drive_sichern** anhaken. Whisper verarbeitet 10-Minuten-Abschnitte und sichert nach jedem fertigen Abschnitt. Dieselbe Datei/Link und dieselbe Modellwahl setzen den Lauf fort. Ohne Drive bleiben Sicherungen nur bis zum Löschen der Laufzeit erhalten. Colab kann Sitzungen beenden; eine unbegrenzte Laufzeit wird nicht zugesichert.

**Falls der Abruf scheitert:** Unter „Technische Details“ stehen die Ergebnisse beider Abrufwege, etwa eine Anfragegrenze, eine Anmeldeanforderung oder ein Verbindungsfehler. Eine solche Meldung bedeutet nicht, dass das Video keine Untertitel hat. Für eine gezielte Fehleranalyse bitte diese Details mitkopieren. Alternativ **Datei_hochladen** wählen und eine eigene Audio-/Videodatei verwenden.

Die Ausgabe ist vollständig und wird nicht zusammengefasst. `transkript.txt` ist zum Lesen und für ChatGPT; `untertitel.srt`/`.vtt` enthalten Zeitangaben. KI-Ergebnisse können Erkennungsfehler enthalten. Kein künstliches Limit für die Videolänge; Laufzeit, Speicher und YouTube-Zugriff bleiben praktische Grenzen.

Nutzt [Erikiss/yt-dlp](https://github.com/Erikiss/yt-dlp), festgelegten Quellstand `897341241f2bfb2439fe0367f4c7c9ccd29a3828`. [Anleitung & technische Details](https://github.com/Erikiss/yt-dlp/blob/subtitle-colab/colab/README.md).


In [ ]:
#@title YouTube-Link einfügen → ▶ starten { display-mode: "form" }
#@markdown **Nur den Link einfügen.** Installation, Untertitelwahl und Download laufen automatisch.
YouTube_Link = "" #@param {type:"string"}
#@markdown Optionale Einstellungen – können unverändert bleiben.
Modus = "Automatisch" #@param ["Automatisch", "Nur vorhandene Untertitel", "KI neu erzeugen"]
Stream_bereits_beendet = False #@param {type:"boolean"}
In_Google_Drive_sichern = False #@param {type:"boolean"}
Datei_hochladen = False #@param {type:"boolean"}
Whisper_Modell = "auto" #@param ["auto", "small", "medium", "turbo", "large-v3"]
ZIP_automatisch_herunterladen = True #@param {type:"boolean"}
#@markdown **Drive:** Ergebnisse und KI-Zwischenstände dauerhaft sichern (Google fragt nach Freigabe).
#@markdown **Stream bereits beendet:** anhaken, falls YouTube eine beendete Aufzeichnung noch als live meldet. Auch `/live/`-Links sind gültig.
#@markdown **Datei hochladen:** eigene Audio-/Videodatei statt Link, z. B. wenn YouTube den Abruf blockiert.
#@markdown **KI:** Sprache bleibt im Original. `auto` nutzt Turbo auf GPU, Small auf CPU. Keine Übersetzung.

_SUBTITLE_CORE = '"""One-link subtitles for Erikiss/yt-dlp; also embedded in the Colab notebook.\n\nNo server, account cookies or API keys. Only standard-library imports until\nyt-dlp / optional faster-whisper are actually needed.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport copy\nimport hashlib\nimport html\nimport json\nimport math\nimport re\nimport shutil\nimport subprocess\nimport tempfile\nimport textwrap\nimport time\nimport zipfile\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime\nfrom pathlib import Path\nfrom urllib.parse import parse_qs, urlparse\n\n\nTOOL_VERSION = 1\nCHUNK_SECONDS = 600\nCONTEXT_SECONDS = 3\n\n\nclass SubtitleError(RuntimeError):\n    """An actionable, user-facing failure."""\n\n    def __init__(self, message, *, details=\'\'):\n        super().__init__(message)\n        self.details = details\n\n\n@dataclass\nclass Cue:\n    start: float\n    end: float\n    text: str\n\n\n@dataclass\nclass Result:\n    title: str\n    source: str\n    language: str\n    directory: Path\n    files: list[Path]\n    archive: Path\n\n\ndef youtube_url(value: str) -> str:\n    """Accept one YouTube video, discard tracking and playlist parameters."""\n    value = value.strip()\n    if not value:\n        raise SubtitleError(\'Bitte zuerst einen YouTube-Link einfügen.\')\n    if re.fullmatch(r\'[A-Za-z0-9_-]{11}\', value):\n        video_id = value\n    else:\n        if value.startswith((\'www.\', \'m.\', \'youtube.com/\', \'youtu.be/\')):\n            value = \'https://\' + value\n        parsed = urlparse(value)\n        host = (parsed.hostname or \'\').lower()\n        if parsed.scheme not in (\'http\', \'https\') or parsed.username or parsed.password:\n            raise SubtitleError(\'Bitte einen gültigen YouTube-Videolink einfügen.\')\n        if host == \'youtu.be\':\n            video_id = parsed.path.strip(\'/\').split(\'/\')[0]\n        elif host in (\'youtube.com\', \'www.youtube.com\', \'m.youtube.com\',\n                      \'music.youtube.com\', \'youtube-nocookie.com\', \'www.youtube-nocookie.com\'):\n            pieces = parsed.path.strip(\'/\').split(\'/\')\n            if parsed.path.rstrip(\'/\') == \'/watch\':\n                video_id = parse_qs(parsed.query).get(\'v\', [\'\'])[0]\n            elif len(pieces) == 2 and pieces[0] in (\'shorts\', \'live\', \'embed\', \'v\'):\n                video_id = pieces[1]\n            else:\n                video_id = \'\'\n        else:\n            raise SubtitleError(\'Dieser Einstieg akzeptiert ausschließlich YouTube-Videolinks.\')\n    if not re.fullmatch(r\'[A-Za-z0-9_-]{11}\', video_id):\n        raise SubtitleError(\'Kein einzelnes Video gefunden. Bitte den Video-Link statt einer Playlist verwenden.\')\n    return f\'https://www.youtube.com/watch?v={video_id}\'\n\n\ndef clean_text(value: str) -> str:\n    return re.sub(r\'\\s+\', \' \', html.unescape(value)).strip()\n\n\ndef parse_json3(data: str) -> list[Cue]:\n    """Keep the full native timed-text stream; no duration or token limit."""\n    events = json.loads(data).get(\'events\', [])\n    cues = []\n    for index, event in enumerate(events):\n        text = clean_text(\'\'.join(segment.get(\'utf8\', \'\') for segment in event.get(\'segs\', [])))\n        if not text or \'tStartMs\' not in event:\n            continue\n        start = max(0, float(event[\'tStartMs\']) / 1000)\n        duration = event.get(\'dDurationMs\')\n        if duration is None:\n            following = next((item[\'tStartMs\'] for item in events[index + 1:]\n                              if item.get(\'tStartMs\', 0) > event[\'tStartMs\']), None)\n            end = float(following) / 1000 if following is not None else start + 3\n        else:\n            end = start + max(0.001, float(duration) / 1000)\n        cues.append(Cue(start, end, text))\n    return normalize_cues(cues)\n\n\ndef parse_timestamp(value: str) -> float:\n    parts = value.replace(\',\', \'.\').split(\':\')\n    if len(parts) == 2:\n        parts.insert(0, \'0\')\n    return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])\n\n\ndef parse_vtt(data: str) -> list[Cue]:\n    cues = []\n    timing = re.compile(r\'(?P<start>(?:\\d+:)?\\d{2}:\\d{2}[.,]\\d+)\\s+-->\\s+\'\n                        r\'(?P<end>(?:\\d+:)?\\d{2}:\\d{2}[.,]\\d+)\')\n    for block in re.split(r\'\\n\\s*\\n\', data.replace(\'\\r\\n\', \'\\n\').lstrip(\'\\ufeff\')):\n        lines = block.splitlines()\n        if not lines or lines[0].startswith((\'NOTE\', \'STYLE\', \'REGION\')):\n            continue\n        for index, line in enumerate(lines):\n            match = timing.match(line)\n            if match:\n                text = clean_text(re.sub(r\'<[^>]+>\', \'\', \' \'.join(lines[index + 1:])))\n                if text:\n                    cues.append(Cue(parse_timestamp(match[\'start\']), parse_timestamp(match[\'end\']), text))\n                break\n    return normalize_cues(cues)\n\n\ndef normalize_cues(cues: list[Cue]) -> list[Cue]:\n    """Remove rolling display repetitions only when their time spans overlap."""\n    output = []\n    previous = None\n    for cue in sorted(cues, key=lambda item: (item.start, item.end)):\n        if not all(math.isfinite(number) for number in (cue.start, cue.end)):\n            continue\n        text = clean_text(cue.text)\n        if not text or cue.end <= cue.start:\n            continue\n        if previous is not None and cue.start < previous.end:\n            old_words, new_words = previous.text.split(), text.split()\n            for count in range(min(len(old_words), len(new_words)), 0, -1):\n                if old_words[-count:] == new_words[:count]:\n                    text = \' \'.join(new_words[count:])\n                    if not text and output:\n                        output[-1].end = max(output[-1].end, cue.end)\n                    break\n        # Compare the next rolling frame with the full previous frame, not the delta.\n        previous = Cue(cue.start, cue.end, clean_text(cue.text))\n        if text:\n            if output and output[-1].end > cue.start:\n                output[-1].end = max(output[-1].start + 0.001, cue.start)\n            output.append(Cue(max(0, cue.start), max(cue.start + 0.001, cue.end), text))\n    return output\n\n\ndef timestamp(seconds: float, *, vtt: bool = False) -> str:\n    milliseconds = max(0, round(seconds * 1000))\n    hours, milliseconds = divmod(milliseconds, 3600000)\n    minutes, milliseconds = divmod(milliseconds, 60000)\n    whole_seconds, milliseconds = divmod(milliseconds, 1000)\n    separator = \'.\' if vtt else \',\'\n    return f\'{hours:02}:{minutes:02}:{whole_seconds:02}{separator}{milliseconds:03}\'\n\n\ndef atomic_text(path: Path, text: str) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.tmp\')\n    temporary.write_text(text, encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef write_exports(directory: Path, cues: list[Cue], metadata: dict, *, partial: bool = False) -> list[Path]:\n    directory.mkdir(parents=True, exist_ok=True)\n    suffix = \'.partial\' if partial else \'\'\n    srt = \'\'.join(f\'{index}\\n{timestamp(cue.start)} --> {timestamp(cue.end)}\\n\'\n                  f\'{textwrap.fill(cue.text, width=84, break_long_words=False, break_on_hyphens=False)}\\n\\n\'\n                  for index, cue in enumerate(cues, 1))\n    vtt = \'WEBVTT\\n\\n\' + \'\'.join(\n        f\'{timestamp(cue.start, vtt=True)} --> {timestamp(cue.end, vtt=True)}\\n\'\n        f\'{html.escape(cue.text, quote=False)}\\n\\n\' for cue in cues)\n    # Paragraphs make multi-hour transcripts readable without discarding words.\n    paragraphs, current, start = [], [], 0.0\n    for cue in cues:\n        if current and cue.start - start >= 60:\n            paragraphs.append(\' \'.join(current))\n            current = []\n        if not current:\n            start = cue.start\n        current.append(cue.text)\n    if current:\n        paragraphs.append(\' \'.join(current))\n    documents = {\n        f\'untertitel{suffix}.srt\': srt,\n        f\'untertitel{suffix}.vtt\': vtt,\n        f\'transkript{suffix}.txt\': \'\\n\\n\'.join(paragraphs) + \'\\n\',\n        f\'transkript_zeitstempel{suffix}.txt\': \'\'.join(\n            f\'[{timestamp(cue.start).split(",")[0]}] {cue.text}\\n\' for cue in cues),\n        f\'info{suffix}.json\': json.dumps({**metadata, \'status\': \'partial\' if partial else \'complete\',\n                                        \'cue_count\': len(cues), \'tool_version\': TOOL_VERSION},\n                                       ensure_ascii=False, indent=2) + \'\\n\',\n    }\n    paths = []\n    for name, content in documents.items():\n        path = directory / name\n        atomic_text(path, content)\n        paths.append(path)\n    return paths\n\n\ndef make_result(directory: Path, cues: list[Cue], metadata: dict, raw: Path | None = None) -> Result:\n    if not cues:\n        raise SubtitleError(\'Keine Sprache erkannt. Es wurde kein leeres Transkript als Erfolg ausgegeben.\')\n    files = write_exports(directory, cues, metadata)\n    if raw is not None:\n        files.append(raw)\n    archive = directory / \'untertitel.zip\'\n    temporary = archive.with_suffix(\'.zip.tmp\')\n    with zipfile.ZipFile(temporary, \'w\', compression=zipfile.ZIP_DEFLATED) as zipped:\n        for path in files:\n            zipped.write(path, arcname=path.name)\n    temporary.replace(archive)\n    return Result(metadata[\'title\'], metadata[\'source\'], metadata.get(\'language\', \'unknown\'), directory, files, archive)\n\n\ndef caption_track(info: dict) -> tuple[str, str, list[dict]] | None:\n    """Prefer human captions in the original language, then original auto captions."""\n    manual = {key: value for key, value in (info.get(\'subtitles\') or {}).items()\n              if key != \'live_chat\' and value}\n    automatic = {key: value for key, value in (info.get(\'automatic_captions\') or {}).items()\n                 if key != \'live_chat\' and value and any(\n                     \'tlang\' not in parse_qs(urlparse(track.get(\'url\', \'\')).query) for track in value)}\n    originals = [key.removesuffix(\'-orig\') for key in automatic if key.endswith(\'-orig\')]\n    language = info.get(\'language\') or (originals[0] if originals else None)\n    if language:\n        for collection, source in ((manual, \'YouTube-Untertitel\'), (automatic, \'YouTube-Autountertitel\')):\n            matches = [key for key in collection\n                       if key.removesuffix(\'-orig\').split(\'-\')[0] == language.split(\'-\')[0]]\n            matches.sort(key=lambda key: (key != language, not key.endswith(\'-orig\'), key))\n            if matches:\n                key = matches[0]\n                return source, key.removesuffix(\'-orig\'), collection[key]\n    if automatic:\n        key = sorted(automatic, key=lambda item: (not item.endswith(\'-orig\'), item))[0]\n        return \'YouTube-Autountertitel\', key.removesuffix(\'-orig\'), automatic[key]\n    if manual:\n        key = sorted(manual, key=lambda item: (item not in (\'en\', \'de\'), item))[0]\n        return \'YouTube-Untertitel (Originalsprache nicht ermittelbar)\', key, manual[key]\n    return None\n\n\ndef error_detail(error: Exception) -> str:\n    """Keep the error type/reason, but never put signed URLs or tokens in reports."""\n    text = getattr(error, \'details\', \'\') or f\'{type(error).__name__}: {error}\'\n    text = re.sub(r\'\\x1b\\[[0-9;]*m\', \'\', text)\n    text = re.sub(r\'https?://[^\\s<>"\\\']+\',\n                  lambda match: urlparse(match[0])._replace(query=\'\', fragment=\'\', netloc=urlparse(match[0]).hostname or \'\').geturl(), text)\n    text = re.sub(r\'(?i)\\b(cookie|authorization|po_token|pot|token|api_key)\\s*[:=]\\s*[^\\s,;]+\', r\'\\1=[entfernt]\', text)\n    return text[:1600]\n\n\ndef youtube_error(error: Exception) -> SubtitleError:\n    if isinstance(error, SubtitleError):\n        return error\n    detail = error_detail(error)\n    lowered = detail.lower()\n    if any(marker in lowered for marker in (\'http error 429\', \'429 client error\', \'rate-limit\', \'too many requests\')):\n        message = \'YouTube meldet für diesen Abruf eine Anfragegrenze (429/Rate-Limit). Bitte später erneut versuchen.\'\n    elif any(marker in lowered for marker in (\'requestblocked\', \'ipblocked\', \'not a bot\', \'sign in\', \'sign-in\', \'captcha\')):\n        message = (\'YouTube verlangt bei diesem Abruf eine Anmeldung oder Botbestätigung. \'\n                   \'Das sagt nichts darüber aus, ob Untertitel vorhanden sind. Ein anderes Tool kann einen anderen Abrufweg verwenden.\')\n    elif any(marker in lowered for marker in (\'http error 403\', \'403 client error\', \'403 forbidden\')):\n        message = \'YouTube lehnt diesen Abruf mit HTTP 403 ab. Die genaue Ursache steht unter „Technische Details“.\'\n    elif \'po token\' in lowered or \'po_token\' in lowered:\n        message = \'Für diesen Abrufweg fehlt ein von YouTube verlangter PO-Token. Details stehen unten.\'\n    elif \'ssl\' in lowered or \'certificate_verify_failed\' in lowered:\n        message = \'Die sichere Verbindung zu YouTube ist fehlgeschlagen (TLS/Zertifikat). Das ist kein Nachweis fehlender Untertitel.\'\n    elif any(marker in lowered for marker in (\'transcriptsdisabled\', \'notranscriptfound\')):\n        message = \'Über diesen Abrufweg wurden keine verfügbaren Untertitel gemeldet.\'\n    elif any(marker in lowered for marker in (\'age-restrict\', \'agerestricted\', \'private\', \'unavailable\', \'not available\', \'removed\')):\n        message = \'YouTube meldet das Video für diesen Abruf als eingeschränkt oder nicht verfügbar. Die genaue Meldung steht unten.\'\n    else:\n        message = f\'Abruf fehlgeschlagen: {detail[:500]}\'\n    return SubtitleError(message, details=detail)\n\n\ndef combined_error(attempts: list[tuple[str, Exception]]) -> SubtitleError:\n    last_error = youtube_error(attempts[-1][1])\n    return SubtitleError(str(last_error), details=\'\\n\\n\'.join(\n        f\'{name}:\\n{error_detail(error)}\' for name, error in attempts))\n\n\ndef player_metadata(payload: dict) -> dict:\n    """Read metadata from the transcript client\'s existing player response."""\n    details = payload.get(\'videoDetails\') or {}\n    microformat = (payload.get(\'microformat\') or {}).get(\'playerMicroformatRenderer\') or {}\n    broadcast = microformat.get(\'liveBroadcastDetails\') or {}\n    ended = False\n    try:\n        ended = datetime.fromisoformat(broadcast[\'endTimestamp\'].replace(\'Z\', \'+00:00\')).timestamp() <= time.time()\n    except (KeyError, TypeError, ValueError):\n        pass\n    live = details.get(\'isLive\', broadcast.get(\'isLiveNow\', False))\n    status = (\'post_live\' if details.get(\'isPostLiveDvr\') else \'was_live\' if ended\n              else \'is_live\' if live else \'is_upcoming\' if details.get(\'isUpcoming\')\n              else \'was_live\' if details.get(\'isLiveContent\') else \'not_live\')\n    try:\n        duration = float(details.get(\'lengthSeconds\') or microformat.get(\'lengthSeconds\') or 0) or None\n    except (TypeError, ValueError):\n        duration = None\n    return {\'title\': details.get(\'title\'), \'duration_seconds\': duration,\n            \'language\': details.get(\'defaultAudioLanguage\'), \'live_status\': status}\n\n\ndef fetch_direct_transcript(canonical: str, root: Path, *, stream_ended=False, progress=print) -> Result:\n    """Try the maintained transcript client before extracting any media formats."""\n    from requests import Session\n    from youtube_transcript_api import YouTubeTranscriptApi\n\n    video_id = parse_qs(urlparse(canonical).query)[\'v\'][0]\n\n    class TranscriptSession(Session):\n        def __init__(self):\n            super().__init__()\n            self.video_info = {}\n\n        def request(self, *args, **kwargs):\n            kwargs.setdefault(\'timeout\', (10, 30))\n            response = super().request(*args, **kwargs)\n            if response.status_code == 200 and urlparse(response.url).path == \'/youtubei/v1/player\':\n                self.video_info = player_metadata(response.json())\n            return response\n\n    with TranscriptSession() as session:\n        transcripts = list(YouTubeTranscriptApi(http_client=session).list(video_id))\n        info = session.video_info\n        if info.get(\'live_status\') in (\'is_live\', \'is_upcoming\') and not stream_ended:\n            raise SubtitleError(\'YouTube meldet dieses Video noch als live. \'\n                                \'Falls es bereits beendet ist, „Stream_bereits_beendet“ anhaken.\')\n        if not transcripts:\n            raise SubtitleError(\'Der direkte Abruf hat keine Untertitelspur geliefert.\')\n        generated = [track for track in transcripts if track.is_generated]\n        language = info.get(\'language\') or (generated[0].language_code if len(generated) == 1 else None)\n        same_language = [track for track in transcripts if track.language_code == language]\n        if language and not same_language:\n            same_language = [track for track in transcripts if track.language_code.split(\'-\')[0] == language.split(\'-\')[0]]\n        selected = next((track for track in same_language if not track.is_generated), None)\n        selected = selected or next(iter(same_language), None) or transcripts[0]\n        source = \'YouTube-Autountertitel\' if selected.is_generated else \'YouTube-Untertitel\'\n        if not same_language:\n            source += \' (Originalsprache nicht ermittelbar)\'\n        progress(f\'Lade {source}, Sprache: {selected.language_code}. Dafür wird keine GPU benötigt.\')\n        fetched = selected.fetch()\n        raw_data = fetched.to_raw_data()\n    cues = normalize_cues([Cue(float(item[\'start\']), float(item[\'start\']) + float(item[\'duration\']), item[\'text\'])\n                           for item in raw_data])\n    if not cues:\n        raise SubtitleError(\'Der direkte Abruf hat eine leere Untertitelspur geliefert.\')\n    directory = root / \'youtube\'\n    raw_path = directory / \'original.transcript-api.json\'\n    atomic_text(raw_path, json.dumps(raw_data, ensure_ascii=False, indent=2))\n    metadata = {**info, \'title\': info.get(\'title\') or video_id, \'url\': canonical, \'video_id\': video_id,\n                \'source\': source, \'language\': selected.language_code, \'backend\': \'youtube-transcript-api\',\n                \'stream_ended_confirmed\': stream_ended, \'reported_live_status\': info.get(\'live_status\')}\n    return make_result(directory, cues, metadata, raw_path)\n\n\ndef is_ended_recording(info: dict) -> bool:\n    """An explicit ended status outranks an inconsistent legacy is_live flag."""\n    return info.get(\'live_status\') in (\'post_live\', \'was_live\') or info.get(\'was_live\') is True\n\n\ndef recording_audio_format(context: dict):\n    """Only choose finite files while a recently ended stream is processing.\n\n    A rolling live manifest can contain only the last part of a long recording.\n    Never silently label that tail as the full transcript or keep recording live.\n    """\n    candidates = [item for item in context.get(\'formats\', [])\n                  if item.get(\'protocol\') in (\'http\', \'https\')\n                  and isinstance(item.get(\'filesize\'), (int, float)) and item[\'filesize\'] > 0\n                  and item.get(\'acodec\') not in (None, \'none\')\n                  and not item.get(\'target_duration\') and not item.get(\'is_from_start\')\n                  and \'incomplete\' not in str(item.get(\'format_note\', \'\')).lower()\n                  and \'last 2 hours\' not in str(item.get(\'format_note\', \'\')).lower()]\n    audio_only = [item for item in candidates if item.get(\'vcodec\') == \'none\']\n    available = audio_only or candidates\n    if not available:\n        raise SubtitleError(\n            \'Der Stream ist beendet bzw. als beendet bestätigt. YouTube liefert hier aber noch keine \'\n            \'vollständige Tonspur als fertige Datei. Die Aufzeichnung kann noch verarbeitet werden. \'\n            \'Bitte später erneut starten; es wird nicht nur das letzte Stück des Streams transkribiert.\')\n    yield available[-1]\n\n\nclass QuietLogger:\n    def debug(self, message):\n        pass\n\n    def info(self, message):\n        pass\n\n    def warning(self, message):\n        # yt-dlp exceptions are handled below; do not overwhelm a phone with logs.\n        pass\n\n    def error(self, message):\n        pass\n\n\ndef youtube_downloader(options: dict):\n    """Use the pinned extractor without irrelevant storyboard processing.\n\n    Its storyboard implementation divides a missing duration by a float. The\n    subtitle workflow does not consume thumbnail/storyboard formats at all.\n    Keep the change local to this downloader instance, including after reruns.\n    """\n    from yt_dlp import YoutubeDL\n    from yt_dlp.extractor.youtube import YoutubeIE\n    from yt_dlp.utils import parse_iso8601\n\n    class SubtitleYoutubeIE(YoutubeIE):\n        @classmethod\n        def ie_key(cls):\n            return YoutubeIE.ie_key()\n\n        def _extract_storyboard(self, player_responses, duration):\n            return iter(())\n\n        def _list_formats(self, video_id, microformats, video_details, player_responses, player_url, duration=None):\n            # A finished broadcast can still carry isLive=True in another client\n            # response. Its recorded end time is stronger evidence of an ending.\n            ended = any(\n                isinstance(item, dict)\n                and (end := parse_iso8601((item.get(\'liveBroadcastDetails\') or {}).get(\'endTimestamp\'))) is not None\n                and end <= time.time()\n                for item in (microformats or []))\n            if ended:\n                video_details = copy.deepcopy(video_details or [{}])\n                for item in video_details:\n                    item.update(isLive=False, isUpcoming=False, isLiveContent=True)\n                microformats = copy.deepcopy(microformats)\n                for item in microformats:\n                    if isinstance(item.get(\'liveBroadcastDetails\'), dict):\n                        item[\'liveBroadcastDetails\'][\'isLiveNow\'] = False\n                # Preserve isPostLiveDvr: yt-dlp needs its distinction between a\n                # processed recording and a recording still being processed.\n            return super()._list_formats(video_id, microformats, video_details,\n                                         player_responses, player_url, duration)\n\n        def raise_no_formats(self, msg, expected=False, video_id=None):\n            # Video playback can fail while a caption track is still returned.\n            # Keep the player failure for a report if no captions can be fetched,\n            # but let upstream finish its metadata/subtitle extraction first.\n            markers = (\'sign in\', \'sign-in\', \'log in\', \'login\', \'not a bot\', \'captcha\',\n                       \'age-restrict\', \'private\', \'unavailable\', \'not available\',\n                       \'removed\', \'rate-limit\', \'too many requests\', \'403\', \'429\', \'po token\')\n            if expected and self.get_param(\'ignore_no_formats_error\') and any(marker in str(msg).lower() for marker in markers):\n                self._downloader.subtitle_playback_errors.append(str(msg))\n            return super().raise_no_formats(msg, expected=expected, video_id=video_id)\n\n    ydl = YoutubeDL(options)\n    ydl.subtitle_playback_errors = []\n    ydl.add_info_extractor(SubtitleYoutubeIE(ydl))\n    return ydl\n\n\ndef ydl_options(*, metadata_only=False) -> dict:\n    runtime = {}\n    if not shutil.which(\'deno\'):\n        try:\n            from deno import find_deno_bin\n            runtime[\'path\'] = str(find_deno_bin())\n        except (ImportError, RuntimeError):\n            pass\n    return {\n        \'noplaylist\': True,\n        \'quiet\': True,\n        \'no_warnings\': True,\n        \'logger\': QuietLogger(),\n        \'socket_timeout\': 30,\n        \'retries\': 3,\n        \'fragment_retries\': 3,\n        \'extractor_retries\': 2,\n        \'extractor_args\': {\'youtube\': {\'skip\': [\'translated_subs\', *([\'hls\', \'dash\'] if metadata_only else [])]}},\n        \'js_runtimes\': {\'deno\': runtime},\n    }\n\n\ndef fetch_caption(ydl, tracks: list[dict]) -> tuple[list[Cue], str, str]:\n    supported = [track for track in tracks if track.get(\'ext\') in (\'json3\', \'vtt\')]\n    supported.sort(key=lambda track: track.get(\'ext\') != \'json3\')\n    if not supported:\n        raise SubtitleError(\'Die Untertitelspur hat kein unterstütztes JSON3- oder VTT-Format.\')\n    from yt_dlp.networking import Request\n\n    last_error = None\n    # Try one URL of each format; avoid hammering duplicate client URLs.\n    formats_seen = set()\n    for track in supported:\n        extension = track[\'ext\']\n        if extension in formats_seen:\n            continue\n        formats_seen.add(extension)\n        try:\n            raw = track.get(\'data\')\n            if raw is None:\n                with ydl.urlopen(Request(track[\'url\'], headers=track.get(\'http_headers\', {}))) as response:\n                    raw = response.read().decode(\'utf-8-sig\')\n            cues = parse_json3(raw) if extension == \'json3\' else parse_vtt(raw)\n            if cues:\n                return cues, raw, extension\n            last_error = SubtitleError(\'YouTube hat eine leere Untertitelspur zurückgegeben.\')\n        except Exception as error:\n            last_error = error\n            if any(code in str(error) for code in (\'429\', \'403\')):\n                break\n    raise youtube_error(last_error or SubtitleError(\'Untertitel konnten nicht gelesen werden.\'))\n\n\ndef audio_duration(path: Path) -> float:\n    if not shutil.which(\'ffprobe\'):\n        raise SubtitleError(\'FFmpeg fehlt. Bitte FFmpeg installieren; in Colab geschieht dies automatisch.\')\n    command = [\'ffprobe\', \'-v\', \'error\', \'-show_entries\', \'format=duration\',\n               \'-of\', \'default=noprint_wrappers=1:nokey=1\', str(path)]\n    result = subprocess.run(command, capture_output=True, text=True, check=False)\n    try:\n        duration = float(result.stdout.strip())\n        if result.returncode or not math.isfinite(duration) or duration <= 0:\n            raise ValueError\n        return duration\n    except ValueError:\n        raise SubtitleError(\'Die Audio-/Videodatei konnte nicht gelesen werden oder hat keine gültige Dauer.\')\n\n\ndef file_digest(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open(\'rb\') as source:\n        for block in iter(lambda: source.read(1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef transcribe_audio(audio: Path, root: Path, metadata: dict, *, model_name=\'auto\', progress=print) -> Result:\n    try:\n        import ctranslate2\n        from faster_whisper import WhisperModel\n    except ImportError:\n        raise SubtitleError(\'Spracherkennung fehlt. Bitte faster-whisper installieren oder die Colab-Startzelle nutzen.\')\n\n    duration = audio_duration(audio)\n    try:\n        has_gpu = ctranslate2.get_cuda_device_count() > 0\n    except RuntimeError:\n        has_gpu = False\n    device, precision = (\'cuda\', \'float16\') if has_gpu else (\'cpu\', \'int8\')\n    if model_name == \'auto\':\n        model_name = \'turbo\' if has_gpu else \'small\'\n    if not has_gpu:\n        progress(\'Spracherkennung läuft auf CPU. Bei langen Videos beschleunigt eine Colab-GPU diesen Schritt.\')\n\n    signature = {\n        \'tool_version\': TOOL_VERSION, \'audio_sha256\': file_digest(audio),\n        \'model\': model_name, \'chunk_seconds\': CHUNK_SECONDS, \'context_seconds\': CONTEXT_SECONDS,\n        \'beam_size\': 5, \'vad_filter\': True, \'word_timestamps\': True,\n    }\n    key = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:16]\n    directory = root / f\'whisper-{key}\'\n    checkpoint = directory / \'fortschritt.json\'\n    state = {\'signature\': signature, \'next_chunk\': 0, \'language\': None, \'cues\': []}\n    if checkpoint.exists():\n        try:\n            saved = json.loads(checkpoint.read_text(encoding=\'utf-8\'))\n            if saved[\'signature\'] != signature:\n                raise ValueError(\'Signatur stimmt nicht überein\')\n            if not isinstance(saved[\'next_chunk\'], int) or saved[\'next_chunk\'] < 0:\n                raise ValueError(\'Ungültiger Fortschritt\')\n            state = saved\n        except (ValueError, KeyError, TypeError) as error:\n            raise SubtitleError(f\'Die Sicherung ist beschädigt: {checkpoint}. Bitte den Ordner umbenennen und neu starten.\') from error\n    count = math.ceil(duration / CHUNK_SECONDS)\n    cues = [Cue(**item) for item in state[\'cues\']]\n    metadata = {**metadata, \'source\': f\'Whisper ({model_name})\', \'duration_seconds\': duration,\n                \'model\': model_name, \'language\': state[\'language\'] or \'unknown\'}\n    if state[\'next_chunk\'] >= count:\n        progress(\'Vollständige Sicherung gefunden – kein erneutes Transkribieren nötig.\')\n        return make_result(directory, cues, metadata)\n\n    progress(f\'Spracherkennung: {model_name} auf {device}; {count} Abschnitte à höchstens 10 Minuten.\')\n    progress(f\'Zwischenstände: {directory}\')\n    if state[\'next_chunk\']:\n        progress(f\'Setze nach {state["next_chunk"]} bereits gesicherten Abschnitten fort.\')\n    try:\n        model = WhisperModel(model_name, device=device, compute_type=precision)\n    except Exception as error:\n        raise SubtitleError(\'Whisper konnte nicht geladen werden. Internetzugriff und freien Speicher prüfen; \'\n                            \'bei CUDA-Fehlern die Colab-Laufzeit neu starten oder CPU wählen. \'\n                            f\'Detail: {str(error)[:300]}\') from error\n\n    with tempfile.TemporaryDirectory(prefix=\'subtitle-chunk-\') as scratch:\n        chunk_file = Path(scratch) / \'chunk.wav\'\n        for index in range(state[\'next_chunk\'], count):\n            start = index * CHUNK_SECONDS\n            stop = min(duration, start + CHUNK_SECONDS)\n            context_start = max(0, start - CONTEXT_SECONDS)\n            context_stop = min(duration, stop + CONTEXT_SECONDS)\n            progress(f\'Abschnitt {index + 1}/{count}: {timestamp(start)} – {timestamp(stop)}\')\n            subprocess.run([\'ffmpeg\', \'-nostdin\', \'-hide_banner\', \'-loglevel\', \'error\', \'-y\',\n                            \'-ss\', str(context_start), \'-i\', str(audio), \'-t\', str(context_stop - context_start),\n                            \'-vn\', \'-ac\', \'1\', \'-ar\', \'16000\', str(chunk_file)], check=True)\n            segments, detected = model.transcribe(\n                str(chunk_file), beam_size=5, language=state[\'language\'], task=\'transcribe\',\n                vad_filter=True, condition_on_previous_text=False, word_timestamps=True,\n            )\n            chunk_cues = []\n            for segment in segments:\n                # Context on both sides protects words at chunk boundaries. Keep\n                # each word only in the interval containing its midpoint.\n                words = [word for word in (segment.words or [])\n                         if start <= context_start + (word.start + word.end) / 2 < stop]\n                if words:\n                    text = clean_text(\'\'.join(word.word for word in words))\n                    if text:\n                        chunk_cues.append(Cue(max(start, context_start + words[0].start),\n                                              min(stop, context_start + words[-1].end), text))\n                elif not segment.words and start <= context_start + (segment.start + segment.end) / 2 < stop:\n                    text = clean_text(segment.text)\n                    if text:\n                        chunk_cues.append(Cue(max(start, context_start + segment.start),\n                                              min(stop, context_start + segment.end), text))\n            cues.extend(cue for cue in chunk_cues if cue.end > cue.start)\n            if chunk_cues and state[\'language\'] is None:\n                state[\'language\'] = detected.language\n            state.update(next_chunk=index + 1, cues=[asdict(cue) for cue in cues])\n            atomic_text(checkpoint, json.dumps(state, ensure_ascii=False))\n            metadata[\'language\'] = state[\'language\'] or \'unknown\'\n            write_exports(directory, cues, {**metadata, \'completed_chunks\': index + 1,\n                                           \'total_chunks\': count}, partial=True)\n            progress(f\'Gesichert: {index + 1}/{count}.\')\n    return make_result(directory, cues, metadata)\n\n\ndef run(url: str = \'\', *, output_dir: str | Path = \'youtube_subtitles\',\n        local_file: str | Path | None = None, mode: str = \'auto\', model_name: str = \'auto\',\n        stream_ended: bool = False, progress=print, transcriber=None) -> Result:\n    if mode not in (\'auto\', \'subtitles\', \'whisper\'):\n        raise SubtitleError(\'Unbekannter Modus.\')\n    if model_name not in (\'auto\', \'small\', \'medium\', \'turbo\', \'large-v3\'):\n        raise SubtitleError(\'Unbekanntes Whisper-Modell.\')\n    output_dir = Path(output_dir).expanduser().resolve()\n    if local_file is not None:\n        audio = Path(local_file).expanduser().resolve()\n        if not audio.is_file():\n            raise SubtitleError(\'Die hochgeladene Datei fehlt.\')\n        if mode == \'subtitles\':\n            raise SubtitleError(\'Für eine eigene Datei bitte „Automatisch“ oder „KI neu erzeugen“ wählen.\')\n        metadata = {\'title\': audio.stem, \'input\': \'local_file\', \'source\': \'Whisper\'}\n        root = output_dir / f\'lokal-{file_digest(audio)[:16]}\'\n    else:\n        canonical = youtube_url(url)\n        video_id = parse_qs(urlparse(canonical).query)[\'v\'][0]\n        root = output_dir / video_id\n        attempts = []\n        if mode != \'whisper\':\n            progress(\'Prüfe Untertitel direkt …\')\n            try:\n                return fetch_direct_transcript(canonical, root, stream_ended=stream_ended, progress=progress)\n            except Exception as error:\n                attempts.append((\'Direkter Untertitelabruf\', error))\n                progress(f\'Direkter Abruf ohne Ergebnis ({type(error).__name__}). Prüfe zusätzlich mit yt-dlp …\')\n        else:\n            progress(\'Prüfe Video für Spracherkennung …\')\n        try:\n            with youtube_downloader({**ydl_options(metadata_only=True), \'skip_download\': True, \'ignore_no_formats_error\': True}) as ydl:\n                info = ydl.extract_info(canonical, download=False)\n                if not info or info.get(\'_type\') in (\'playlist\', \'multi_video\'):\n                    raise SubtitleError(\'Kein einzelnes Video gefunden.\')\n                ended = is_ended_recording(info)\n                reports_live = info.get(\'is_live\') or info.get(\'live_status\') in (\'is_live\', \'is_upcoming\')\n                if reports_live and not ended and not stream_ended:\n                    raise SubtitleError(\n                        \'YouTube meldet dieses Video noch als live oder angekündigt. \'\n                        \'Falls der Stream bereits beendet ist: „Stream_bereits_beendet“ anhaken und erneut starten. \'\n                        \'Der /live/-Link ist auch für Aufzeichnungen gültig.\')\n                cautious_recording = info.get(\'live_status\') == \'post_live\' or bool(reports_live and (ended or stream_ended))\n                metadata = {\'title\': info.get(\'title\') or video_id, \'url\': canonical,\n                            \'video_id\': video_id, \'duration_seconds\': info.get(\'duration\'),\n                            \'reported_live_status\': info.get(\'live_status\'), \'stream_ended_confirmed\': stream_ended}\n                progress(metadata[\'title\'])\n                if cautious_recording:\n                    progress(\'Beendete Aufzeichnung: prüfe Untertitel und bereits fertig bereitgestellte Audiodateien.\')\n                track = caption_track(info)\n                if mode != \'whisper\' and track:\n                    source, language, formats = track\n                    progress(f\'Lade {source}, Sprache: {language}. Dafür wird keine GPU benötigt.\')\n                    try:\n                        cues, raw, extension = fetch_caption(ydl, formats)\n                    except Exception as error:\n                        attempts.append((\'yt-dlp: Untertiteldatei\', error))\n                        progress(\'Die Untertiteldatei konnte nicht geladen werden.\')\n                    else:\n                        directory = root / \'youtube\'\n                        raw_path = directory / f\'original.{extension}\'\n                        atomic_text(raw_path, raw)\n                        return make_result(directory, cues, {**metadata, \'source\': source, \'language\': language,\n                                                            \'backend\': \'yt-dlp\'}, raw_path)\n                playback_errors = getattr(ydl, \'subtitle_playback_errors\', [])\n                if playback_errors:\n                    raise RuntimeError(\'\\n\'.join(playback_errors))\n        except Exception as error:\n            attempts.append((\'yt-dlp: Videometadaten\', error))\n            raise combined_error(attempts) from error\n        if mode == \'subtitles\':\n            if track:\n                raise combined_error(attempts)\n            if ended or stream_ended:\n                message = (\'Für diese Aufzeichnung hat keiner der Abrufwege Untertitel geliefert. \'\n                           \'Sie können noch verarbeitet werden oder hier nicht abrufbar sein. \'\n                           \'„Automatisch“ versucht zusätzlich die Tonspur.\')\n            else:\n                message = \'Keiner der Abrufwege hat vorhandene Untertitel geliefert. Für Spracherkennung „Automatisch“ wählen.\'\n            raise SubtitleError(message, details=combined_error(attempts).details if attempts else \'\')\n        progress(\'Lade nur die Tonspur für die Spracherkennung …\')\n        cache = output_dir / \'_audio\' / video_id\n        cache.mkdir(parents=True, exist_ok=True)\n        try:\n            require_finite_audio = cautious_recording or stream_ended\n\n            def avoid_live_recording(latest_info, *, incomplete=False):\n                nonlocal require_finite_audio\n                latest_live = latest_info.get(\'is_live\') or latest_info.get(\'live_status\') in (\'is_live\', \'is_upcoming\')\n                if latest_live and not is_ended_recording(latest_info) and not stream_ended:\n                    raise SubtitleError(\'YouTube meldet beim Audioabruf erneut einen Live-Status. \'\n                                        \'Bei einer beendeten Sendung „Stream_bereits_beendet“ aktivieren.\')\n                if latest_info.get(\'live_status\') == \'post_live\' or latest_live:\n                    require_finite_audio = True\n\n            def select_audio(context):\n                if require_finite_audio:\n                    yield from recording_audio_format(context)\n                else:\n                    yield from ydl.build_format_selector(\'bestaudio/best\')(context)\n\n            with youtube_downloader({**ydl_options(),\n                                     \'format\': select_audio,\n                                     \'match_filter\': avoid_live_recording,\n                                     \'skip_unavailable_fragments\': False,\n                                     \'outtmpl\': str(cache / \'audio.%(ext)s\')}) as ydl:\n                audio_info = ydl.extract_info(canonical, download=True)\n                audio = Path(ydl.prepare_filename(audio_info))\n                if not audio.is_file():\n                    raise SubtitleError(\'Die Tonspur wurde nicht vollständig heruntergeladen.\')\n                expected_duration = audio_info.get(\'duration\') or metadata.get(\'duration_seconds\')\n                if (ended or stream_ended or is_ended_recording(audio_info)) and isinstance(expected_duration, (int, float)) and expected_duration > 0:\n                    actual_duration = audio_duration(audio)\n                    if actual_duration < expected_duration - max(30, expected_duration * 0.02):\n                        # Do not keep reusing a cached tail after YouTube finishes\n                        # processing. Keep it separately for inspection instead.\n                        audio.replace(audio.with_name(f\'audio.incomplete{audio.suffix}\'))\n                        raise SubtitleError(\'Die geladene Tonspur ist deutlich kürzer als die Aufzeichnung. \'\n                                            \'YouTube hat möglicherweise noch nicht alles bereitgestellt. \'\n                                            \'Bitte später erneut starten; kein vollständiges Transkript erzeugt.\')\n        except Exception as error:\n            attempts.append((\'yt-dlp: Tonspur\', error))\n            raise combined_error(attempts) from error\n    return (transcriber or transcribe_audio)(audio, root, metadata, model_name=model_name, progress=progress)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=\'YouTube-Link → SRT, VTT und vollständiges TXT-Transkript\')\n    parser.add_argument(\'url\', nargs=\'?\', default=\'\')\n    parser.add_argument(\'--file\', dest=\'local_file\', help=\'Eigene Audio-/Videodatei statt YouTube\')\n    parser.add_argument(\'--output\', default=\'youtube_subtitles\')\n    parser.add_argument(\'--mode\', choices=(\'auto\', \'subtitles\', \'whisper\'), default=\'auto\')\n    parser.add_argument(\'--model\', choices=(\'auto\', \'small\', \'medium\', \'turbo\', \'large-v3\'), default=\'auto\')\n    parser.add_argument(\'--stream-ended\', action=\'store_true\', help=\'Bestätigt: Der Livestream ist bereits beendet\')\n    args = parser.parse_args()\n    try:\n        result = run(args.url, local_file=args.local_file, output_dir=args.output,\n                     mode=args.mode, model_name=args.model, stream_ended=args.stream_ended)\n        print(f\'Fertig: {result.archive}\')\n    except SubtitleError as error:\n        parser.exit(1, f\'{error}\\n\' + (f\'\\nTechnische Details:\\n{error.details}\\n\' if error.details else \'\'))\n    except KeyboardInterrupt:\n        parser.exit(130, \'Gestoppt. Bereits gesicherte Abschnitte bleiben erhalten.\\n\')\n'
exec(compile(_SUBTITLE_CORE, "subtitle_tool.py", "exec"))

"""Colab-only launcher. Embedded after subtitle_tool.py by build_notebook.py."""

# This file is combined with the core, so its imports and classes are available.
import base64
import importlib
import os
import sys


def _install(*packages):
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                             '--disable-pip-version-check', *packages],
                            capture_output=True, text=True)
    if result.returncode:
        raise SubtitleError('Installation fehlgeschlagen. Bitte die Startzelle erneut ausführen.\n'
                            + (result.stderr or result.stdout)[-1800:])
    importlib.invalidate_caches()


def _setup_ytdlp():
    global _subtitle_setup_version
    commit = '897341241f2bfb2439fe0367f4c7c9ccd29a3828'
    version = (commit, '1.2.4', '1.3')
    if globals().get('_subtitle_setup_version') != version:
        print('Einmalige Einrichtung: direkter Untertitelabruf und yt-dlp aus deinem Repo …', flush=True)
        _install('youtube-transcript-api==1.2.4',
                 f'yt-dlp[default,deno] @ git+https://github.com/Erikiss/yt-dlp.git@{commit}')
        # The notebook may have previously imported another yt-dlp release.
        for name in list(sys.modules):
            if name in ('yt_dlp', 'youtube_transcript_api') or name.startswith(('yt_dlp.', 'youtube_transcript_api.')):
                del sys.modules[name]
        _subtitle_setup_version = version


def _setup_ffmpeg():
    if shutil.which('ffmpeg') and shutil.which('ffprobe'):
        return
    print('Richte Audioverarbeitung ein …', flush=True)
    for arguments in (['apt-get', '-qq', 'update'], ['apt-get', '-qq', 'install', '-y', 'ffmpeg']):
        subprocess.run(arguments, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)


def _colab_transcriber(audio, root, metadata, *, model_name, progress):
    global _subtitle_whisper_ready
    _setup_ffmpeg()
    # CUDA libraries must be on LD_LIBRARY_PATH BEFORE the worker starts.
    # The worker also isolates native imports from Colab's preloaded packages.
    if not globals().get('_subtitle_whisper_ready'):
        progress('Richte Spracherkennung ein (nur beim ersten Bedarf) …')
        _install('faster-whisper>=1.2,<2')
        gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True).returncode == 0 if shutil.which('nvidia-smi') else False
        if gpu:
            _install('nvidia-cublas-cu12>=12,<13', 'nvidia-cudnn-cu12>=9,<10')
        _subtitle_whisper_ready = True
    environment = dict(os.environ)
    library_dirs = []
    for location in sys.path:
        if location:
            library_dirs.extend(str(path) for path in Path(location).glob('nvidia/*/lib') if path.is_dir())
    environment['LD_LIBRARY_PATH'] = ':'.join(library_dirs + [environment.get('LD_LIBRARY_PATH', '')])
    worker_directory = Path(tempfile.mkdtemp(prefix='subtitle-worker-'))
    worker = worker_directory / 'worker.py'
    job = worker_directory / 'job.json'
    result_path = worker_directory / 'result.json'
    job.write_text(json.dumps({'audio': str(audio), 'root': str(root), 'metadata': metadata,
                               'model_name': model_name}), encoding='utf-8')
    worker.write_text(_SUBTITLE_CORE + '''
import sys
job = json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
result = transcribe_audio(Path(job['audio']), Path(job['root']), job['metadata'], model_name=job['model_name'])
Path(sys.argv[2]).write_text(json.dumps({
    'title': result.title, 'source': result.source, 'language': result.language,
    'directory': str(result.directory), 'files': [str(p) for p in result.files], 'archive': str(result.archive)
}), encoding='utf-8')
''', encoding='utf-8')
    process = subprocess.Popen([sys.executable, '-u', str(worker), str(job), str(result_path)],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=environment)
    recent = []
    try:
        for line in process.stdout:
            recent.append(line.rstrip())
            recent = recent[-12:]
            progress(line.rstrip())
        if process.wait() != 0:
            raise SubtitleError('Die Spracherkennung wurde unterbrochen. Gesicherte Abschnitte bleiben erhalten. '
                                'Bei CUDA-Fehlern die Laufzeit neu starten oder auf CPU wechseln.\n'
                                + '\n'.join(recent[-5:]))
        data = json.loads(result_path.read_text(encoding='utf-8'))
        return Result(data['title'], data['source'], data['language'], Path(data['directory']),
                      [Path(path) for path in data['files']], Path(data['archive']))
    finally:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        if process.stdout:
            process.stdout.close()
        shutil.rmtree(worker_directory, ignore_errors=True)


def _show_result(result):
    from IPython.display import HTML, display
    from google.colab import files

    print(f'Fertig · {result.source} · Sprache: {result.language}', flush=True)
    print(f'Gespeichert unter: {result.directory}', flush=True)
    links = []
    for path in [result.archive, *result.files]:
        if path.name.endswith('.json') or path.name.startswith('original.'):
            continue
        mime = 'application/zip' if path.suffix == '.zip' else 'text/plain;charset=utf-8'
        data = base64.b64encode(path.read_bytes()).decode('ascii')
        label = {'untertitel.zip': 'Alles als ZIP', 'untertitel.srt': 'Untertitel · SRT',
                 'untertitel.vtt': 'Untertitel · VTT', 'transkript.txt': 'Transkript · TXT',
                 'transkript_zeitstempel.txt': 'Text mit Zeitstempeln'}.get(path.name, path.name)
        links.append(f'<a download="{html.escape(path.name)}" href="data:{mime};base64,{data}" '
                     'style="display:inline-block;padding:12px 16px;margin:5px;border-radius:9px;'
                     'background:#155e75;color:white;text-decoration:none;font:15px sans-serif">'
                     f'{label}</a>')
    display(HTML('<div style="padding:12px 0">' + ''.join(links) + '</div>'))
    print('Falls ein Download auf dem Handy nicht öffnet: links im Colab-Dateibereich die Datei herunterladen.')
    if ZIP_automatisch_herunterladen:
        try:
            files.download(str(result.archive))
        except Exception:
            print('Automatischer Download nicht möglich. Bitte einen Downloadknopf oder den Dateibereich verwenden.')


def _start_colab():
    from google.colab import files

    print('Untertitel-Tool · Version 1.3', flush=True)
    if not Datei_hochladen:
        youtube_url(YouTube_Link)  # Fail before installing anything for an empty link.
        _setup_ytdlp()
    output = Path('/content/YouTube_Untertitel')
    if In_Google_Drive_sichern:
        from google.colab import drive
        drive.mount('/content/drive')
        output = Path('/content/drive/MyDrive/YouTube_Untertitel')
    local_file = None
    if Datei_hochladen:
        print('Bitte genau eine eigene Audio- oder Videodatei wählen.', flush=True)
        upload_directory = Path('/content/YouTube_Uploads')
        upload_directory.mkdir(parents=True, exist_ok=True)
        previous_directory = Path.cwd()
        try:
            os.chdir(upload_directory)
            uploaded = files.upload()
            if len(uploaded) != 1:
                raise SubtitleError('Bitte genau eine Datei hochladen und erneut starten.')
            local_file = upload_directory / next(iter(uploaded))
            del uploaded  # Do not keep the upload's bytes in memory during inference.
        finally:
            os.chdir(previous_directory)
    mode = {'Automatisch': 'auto', 'Nur vorhandene Untertitel': 'subtitles', 'KI neu erzeugen': 'whisper'}[Modus]
    if mode != 'subtitles':
        _setup_ffmpeg()  # Also needed to check the duration of a fresh recording.
    result = run(YouTube_Link, local_file=local_file, output_dir=output, mode=mode,
                 model_name=Whisper_Modell, stream_ended=Stream_bereits_beendet, transcriber=_colab_transcriber,
                 progress=lambda message: print(message, flush=True))
    _show_result(result)


def _show_error(message, details=''):
    # Colab's plain preformatted output can be clipped on a narrow phone screen.
    from IPython.display import HTML, display

    diagnostic = ('<details open style="margin-top:12px"><summary>Technische Details – bei Rückfragen mitkopieren</summary>'
                  '<pre style="white-space:pre-wrap;overflow-wrap:anywhere;font:12px/1.5 monospace">'
                  + html.escape(details) + '</pre></details>') if details else ''
    display(HTML('<div role="alert" style="white-space:pre-wrap;overflow-wrap:anywhere;'
                 'padding:14px;border:1px solid #b45309;border-radius:9px;'
                 'font:15px/1.5 sans-serif">' + html.escape(message) + diagnostic + '</div>'))


try:
    _start_colab()
except SubtitleError as error:
    _show_error(f'Hinweis: {error}', error.details)
except KeyboardInterrupt:
    print('\nGestoppt. Bereits gesicherte Abschnitte bleiben erhalten. Zum Fortsetzen dieselben Einstellungen nutzen.')
except Exception as error:
    _show_error(f'Der Lauf konnte nicht abgeschlossen werden: {type(error).__name__}: {error}\n'
                'Es wurde kein vollständiges Ergebnis bestätigt. Gesicherte .partial-Dateien bleiben erhalten.')
